In [2]:
import pandas as pd 
import mysql.connector

In [3]:
# konekcija
conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="statsuser",
    password="statspass",
    database="statsdb"
)

In [4]:
query = """
SELECT cusRef as userId, devRef as devId, name as channel, duration, extra, insertedTS
FROM statistic
WHERE type = 'LiveUsage'
"""

df = pd.read_sql(query, conn)

C:\Users\radet\AppData\Local\Temp\ipykernel_14120\1602444335.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13231016 entries, 0 to 13231015
Data columns (total 6 columns):
 #   Column      Dtype         
---  ------      -----         
 0   userId      str           
 1   devId       str           
 2   channel     str           
 3   duration    int64         
 4   extra       str           
 5   insertedTS  datetime64[us]
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 605.7 MB


In [7]:
import json

df_small = df[['devId', 'extra']]  # samo što ti treba

# prvo pretvori u dict (jedan prolaz)
extra_parsed = df_small['extra'].apply(lambda x: json.loads(x) if x else {})

# onda normalizuj
extra_df = pd.json_normalize(extra_parsed)

# rename
extra_df = extra_df.rename(columns={
    'duration': 'program_duration',
    'programId': 'program_id',
    'startTime': 'start_time'
})

# spoji
df_parsed = pd.concat([df, extra_df], axis=1)

In [ ]:
df_parsed.isna().sum()

userId                     0
devId                      0
channel                    0
duration                   0
extra                      0
insertedTS                 0
title                  55911
genre                      0
start_time                 0
program_id             41670
program_duration         165
location                7748
programUUID         13230851
dtype: int64

In [9]:
df_parsed = df_parsed.drop(columns=['programUUID'])
df_parsed = df_parsed.dropna()

In [47]:
df_parsed = df_parsed.drop_duplicates()

In [10]:
# čišćenje
df_parsed = df_parsed[
    (df_parsed['title'] != 'No title') &
    (df_parsed['duration'] <= 350000)
]

In [68]:
df_clean = df_parsed.drop(columns=['extra'])
df_clean[['program_id', 'program_duration']] = df_clean[['program_id', 'program_duration']].astype('int64')

In [69]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 13124679 entries, 0 to 13231015
Data columns (total 11 columns):
 #   Column            Dtype         
---  ------            -----         
 0   userId            str           
 1   devId             str           
 2   channel           str           
 3   duration          int64         
 4   insertedTS        datetime64[us]
 5   title             str           
 6   genre             str           
 7   start_time        int64         
 8   program_id        int64         
 9   program_duration  int64         
 10  location          str           
dtypes: datetime64[us](1), int64(4), str(6)
memory usage: 1.2 GB


In [70]:
#vrijeme gledanja kanala
df_watch_time = (
    df_clean
    .groupby(['userId', 'devId', 'channel'], as_index=False)
    .agg(total_watch_time=('duration', 'sum'))
)

df_watch_time['total_watch_time'] = (df_watch_time['total_watch_time'] / 1000).round(2)

df_watch_time

,userId,devId,channel,total_watch_time
0,1000,10012538,ATV HD,2379.02
1,1000,10012538,Arena Sport 4,4441.33
2,1000,10012538,Euronews Serbia HD,90.05
3,1000,10012538,Eurosport 1,5040.45
4,1000,10012538,INFO,893.81
...,...,...,...,...
688726,9999999594319999,99999995943199991,Pink Crime & Mystery,1786.16
688727,9999999594319999,99999995943199991,Pink Premium,59355.47
688728,9999999594319999,99999995943199991,Pink Sci-Fi & Fantasy,3967.45
688729,9999999594319999,99999995943199991,Pink Western,19805.66


In [71]:
top_channel = (
    df_watch_time
    .groupby('channel')['devId']
    .nunique()
    .reset_index(name='device_count')
    .sort_values(by='device_count', ascending=False)
)

top_channel

,channel,device_count
62,BN HD,29083
346,RTRS,27414
4,ATV HD,19238
329,Prva,17974
406,SUPERSTAR HD,17639
...,...,...
170,Gametoon HD,2
154,Fast&Fun Box HD,1
291,Pink Folk 1,1
416,TDC HD,1


In [65]:
top_titles_filter = top_channel[(top_channel['channel'] == "Elita Live 1 HD")]
top_titles_filter

,channel,device_count
112,Elita Live 1 HD,66


In [29]:
avg_watch = (
    df_watch_time
    .groupby('channel')['total_watch_time']
    .mean()
    .reset_index(name='avg_watch_time')
    .sort_values(by='avg_watch_time', ascending=False)
)
avg_watch['avg_watch_time'] = (avg_watch['avg_watch_time']).round(2)

avg_watch

,channel,avg_watch_time
112,Elita Live 1 HD,34760.50
116,Elita live 1,23336.14
55,BHT HD,19793.44
182,HGTV HD,18232.38
229,Kontakt radio,17701.28
...,...,...
170,Gametoon HD,99.18
322,Playboy TV HD,80.79
345,RTR Planeta,50.35
122,EroXXX HD,40.93


In [30]:
df_tmp = df_clean[['devId', 'channel', 'program_id', 'title']]

df_unique = df_tmp.drop_duplicates()

top_titles = (
    df_unique
    .groupby(['program_id', 'channel', 'title'])['devId']
    .count()
    .reset_index(name='viewer_count')
    .sort_values(by='viewer_count', ascending=False)
)

top_titles

,program_id,channel,title,viewer_count
7067,68462975,BN HD,Dnevnik 2,14428
14251,68593656,BN HD,Crno na bijelo,12814
7069,68462976,BN HD,Zarobljena,12028
13083,68586920,RTRS,Dnevnik 2,11045
7050,68462962,BN HD,"Dobro jutro, Srpska!",10025
...,...,...,...,...
14,68401723,Disney Channel,Kif,1
13,68401722,Disney Channel,Finis i Ferb,1
12,68401721,Disney Channel,Finis i Ferb,1
11,68401720,Disney Channel,Mirakulus,1


In [ ]:
df_user = (
    df_clean
    .groupby(['devId', 'program_id', 'channel', 'title'], as_index=False)
    .agg(user_watch_time=('duration', 'sum'))
)

df_avg_channel_watch = (
    df_user
    .groupby(['program_id', 'channel', 'title'], as_index=False)
    .agg(avg_watch_time=('user_watch_time', 'mean'))
)

# dodaj program_duration posebno
df_avg_channel_watch = df_avg_channel_watch.merge(
    df_clean[['program_id', 'program_duration']].drop_duplicates(),
    on='program_id',
    how='left'
)

df_avg_channel_watch['avg_watch_time'] = df_avg_channel_watch['avg_watch_time'].round(2)

df_avg_channel_watch = df_avg_channel_watch.sort_values(by='avg_watch_time', ascending=False)

df_avg_channel_watch

,program_id,channel,title,avg_watch_time,program_duration
11231,68524951,HAYAT HD,387 playlista,13866837.37,16560000
12933,68581098,Pink BH 1,Elita 9,13081848.79,21420000
12932,68581098,Pink BH 1,Elita 9,13081848.79,21600000
12048,68565583,Pink BH,Elita 9,11696442.12,21600000
12049,68565583,Pink BH,Elita 9,11696442.12,21420000
...,...,...,...,...,...
12073,68565690,Pink HA HA,Leteći Cirkus Monti Pajtona,10145.00,1860000
5363,68451008,Rai 2 HD,Rai TG Sport Sera,10137.00,1380000
2983,68448312,France 24,Actuelles,10134.00,900000
3762,68449561,LH TV HD,Must See,10119.00,180000


In [ ]:
df_avg_channel_watch.info()

<class 'pandas.DataFrame'>
Index: 15629 entries, 11231 to 8598
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   program_id        15629 non-null  Int64  
 1   channel           15629 non-null  str    
 2   title             15629 non-null  str    
 3   avg_watch_time    15629 non-null  float64
 4   program_duration  15629 non-null  Int64  
dtypes: Int64(2), float64(1), str(2)
memory usage: 763.1 KB


In [ ]:
df_filtered = df_avg_channel_watch[
    (df_avg_channel_watch['avg_watch_time'] >= 0.9 * df_avg_channel_watch['program_duration'])
]

df_filtered = df_filtered.sort_values(by='avg_watch_time', ascending=False)

df_filtered

,program_id,channel,title,avg_watch_time,program_duration
8544,68482862,O Kanal Music,Trap music,10806287.00,10800000
9505,68490630,Arena Esports HD,ESL Pro Liga,10803534.75,10800000
8003,68479804,DM SAT,SMS CHAT,10202940.67,10200000
11492,68526693,TVE,La noche en 24 horas,9001720.00,9000000
9354,68488879,Hayat Folk Box HD,Samo naše,8704455.00,8700000
...,...,...,...,...,...
12275,68572906,Adria TV,Saobraćaj,55456.00,60000
12327,68572958,Adria TV,Vrijeme,55393.89,60000
5917,68451715,Russia 24,No comments,55286.70,60000
12294,68572925,Adria TV,Na današnji dan,54621.29,60000
